# Creative Plots for Wish Simulation

本笔记展示三种更有表现力的可视化：ECDF/Survival、Violin+Strip、累计概率曲线。
可在下方修改 `runs` / `seed` / `outdir`，执行生成图像文件。


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_RATE = 0.016
PITY = 90


def simulate_one_cycle():
    count = 0
    while True:
        count += 1
        if count >= PITY:
            return count
        if np.random.random() < BASE_RATE:
            return count


def simulate_basic(n_runs: int):
    data = [simulate_one_cycle() for _ in range(n_runs)]
    arr = np.array(data)
    return arr, float(np.mean(arr)), float(np.std(arr, ddof=1))


def plot_ecdf_and_survival(data: np.ndarray, out_path: str):
    sns.set(style="whitegrid")
    sorted_data = np.sort(data)
    n = len(sorted_data)
    ecdf_y = np.arange(1, n + 1) / n
    survival_y = 1 - ecdf_y

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.step(sorted_data, ecdf_y, where="post", color="steelblue", label="ECDF (P(X≤x))")
    ax.step(sorted_data, survival_y, where="post", color="tomato", linestyle="--", label="Survival (P(X>x))")
    ax.fill_between(sorted_data, ecdf_y, step="post", alpha=0.1, color="steelblue")
    ax.fill_between(sorted_data, survival_y, step="post", alpha=0.08, color="tomato")
    ax.set_xlim(0, 95)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Number of Wishes")
    ax.set_ylabel("Probability")
    ax.set_title("Cumulative vs Survival Probability of Getting a 5-Star")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=220)
    plt.close(fig)


def plot_violin_swarm(data: np.ndarray, out_path: str):
    sns.set(style="whitegrid")
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.violinplot(x=data, inner=None, color="#c7e9c0", ax=ax)
    sample_n = min(len(data), 1500)
    sample = np.random.choice(data, size=sample_n, replace=False)
    sns.stripplot(x=sample, size=2, color="#2b8cbe", alpha=0.35, jitter=0.25, ax=ax)
    ax.set_xlim(-5, 95)
    ax.set_xlabel("Number of Wishes")
    ax.set_yticks([])
    ax.set_title("Density + Discrete Scatter of Wishes Needed")
    fig.tight_layout()
    fig.savefig(out_path, dpi=220)
    plt.close(fig)


def plot_cumulative_odds(data: np.ndarray, out_path: str):
    sns.set(style="whitegrid")
    max_n = 90
    counts = np.zeros(max_n)
    for v in data:
        if v <= max_n:
            counts[v - 1] += 1
    cumulative = np.cumsum(counts) / len(data)
    n_axis = np.arange(1, max_n + 1)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(n_axis, cumulative, color="#1d91c0", lw=2, label="P(get 5-star by N)")
    ax.axhline(0.5, color="#fb6a4a", ls="--", lw=1.5, label="50% point")
    ax.axvline(n_axis[np.searchsorted(cumulative, 0.5)], color="#fb6a4a", ls=":", lw=1)
    ax.fill_between(n_axis, cumulative, alpha=0.15, color="#1d91c0")
    ax.set_xlim(0, 95)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Number of Wishes (N)")
    ax.set_ylabel("Cumulative Probability")
    ax.set_title("Cumulative Odds of Obtaining a 5-Star by Pull N")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=220)
    plt.close(fig)


In [ ]:
# 运行参数：调整后执行本单元
runs = 50000
seed = 2000
outdir = "figures_notebook"

if seed is not None:
    np.random.seed(seed)

os.makedirs(outdir, exist_ok=True)

# 生成并保存三类创意图
sns.set(style="whitegrid")
data, mean_val, std_val = simulate_basic(runs)
print(f"Runs: {runs}")
print(f"Mean wishes: {mean_val:.3f}")
print(f"Std wishes: {std_val:.3f}")

plot_ecdf_and_survival(data, os.path.join(outdir, "ecdf_survival.png"))
plot_violin_swarm(data, os.path.join(outdir, "violin_swarm.png"))
plot_cumulative_odds(data, os.path.join(outdir, "cumulative_odds.png"))
print(f"Saved creative figures to: {outdir}")
